# Project 2: Wedding planner

In [9]:
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
import asyncio
import os
from pprint import pprint
from typing import Any, Dict

from dotenv import find_dotenv, load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent
from tavily import TavilyClient

# Load environment variables
load_dotenv(find_dotenv())

print("✅ Libraries and environment loaded!")

✅ Libraries and environment loaded!


In [11]:
class RetryMCPInterceptor:
    """Intercepts MCP tool calls: retries transient network failures with exponential backoff."""

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                if exc.error.code not in {-32603}:
                    return CallToolResult(
                        content=[
                            TextContent(
                                type="text",
                                text=f"Tool error (non-retryable): {exc}",
                            )
                        ],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2**attempt)

        return CallToolResult(
            content=[
                TextContent(
                    type="text",
                    text=f"Tool failed after {self.max_retries} attempts: {last_error}",
                )
            ],
            isError=False,
        )


# Connect to Kiwi Travel MCP over HTTP with the retry interceptor
mcp_client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com",
        }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

travel_tools = await mcp_client.get_tools()
print(f"✈️ Loaded {len(travel_tools)} Kiwi flight tools with Retry Interceptor!")

✈️ Loaded 2 Kiwi flight tools with Retry Interceptor!


In [12]:
# 1. Bounded Tavily Web Search Tool
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Any:
    """Useful for searching the web for venues, hotels, and music playlists.
    Pass natural language keywords.
    """
    try:
        return tavily_client.search(query=query, max_results=3)
    except Exception as e:
        return f"Search error for '{query}': {str(e)}"


# 2. Central State Schema
class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    vibe: str
    music_genre: str


print("🔍 Tavily Search tool and WeddingState schema ready!")

🔍 Tavily Search tool and WeddingState schema ready!


In [13]:
# Base Model
model = init_chat_model(
    model="models/gemini-3.5-flash-lite", model_provider="google_genai"
)

# 1. Travel Subagent
travel_subagent = create_agent(
    model=model,
    tools=travel_tools,
    system_prompt=(
        "You are an expert flight booking specialist. Search for the best flight routes, "
        "prices, and schedules for the couple and guests. Return a concise shortlist with prices. "
        "Do not ask follow up questions."
    ),
)

# 2. Venue Subagent
venue_subagent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=(
        "You are an elite wedding venue scout. Use web search to find 2-3 stunning wedding venues "
        "matching the requested destination, guest count, and aesthetic. Do not make up venues."
    ),
)

# 3. DJ Subagent
dj_subagent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=(
        "You are a professional wedding DJ. Curate a 3-act playlist (Ceremony, Cocktail Hour, Dance Party) "
        "tailored to the requested genre and vibe with iconic track names."
    ),
)

print("👥 All 3 Subagents created!")

👥 All 3 Subagents created!


In [14]:
@tool
async def search_flights(runtime: ToolRuntime[WeddingState]) -> str:
    """Travel agent searches for flights based on state origin and destination."""
    origin = runtime.state.get("origin", "London")
    destination = runtime.state.get("destination", "Naples")
    query = f"Find the best direct or 1-stop flights from {origin} to {destination}"
    res = await travel_subagent.ainvoke(
        {"messages": [HumanMessage(content=query)]}
    )
    ans = res["messages"][-1].content
    return ans[0]["text"] if isinstance(ans, list) else str(ans)


@tool
async def search_venues(runtime: ToolRuntime[WeddingState]) -> str:
    """Venue agent finds top venues based on state destination, guest count, and vibe."""
    destination = runtime.state.get("destination", "Amalfi Coast")
    guests = runtime.state.get("guest_count", "40")
    vibe = runtime.state.get("vibe", "cliffside luxury")
    query = f"Find top wedding venues in {destination} for {guests} guests with a {vibe} style"
    res = await venue_subagent.ainvoke(
        {"messages": [HumanMessage(content=query)]}
    )
    ans = res["messages"][-1].content
    return ans[0]["text"] if isinstance(ans, list) else str(ans)


@tool
async def suggest_playlist(runtime: ToolRuntime[WeddingState]) -> str:
    """DJ agent curates playlists based on state music genre and vibe."""
    genre = runtime.state.get(
        "music_genre", "acoustic pop and 80s dance Italian lounge"
    )
    vibe = runtime.state.get("vibe", "cliffside luxury celebration")
    query = f"Curate a wedding ceremony and reception playlist for genre: {genre}, vibe: {vibe}"
    res = await dj_subagent.ainvoke(
        {"messages": [HumanMessage(content=query)]}
    )
    ans = res["messages"][-1].content
    return ans[0]["text"] if isinstance(ans, list) else str(ans)


@tool
def update_wedding_state(
    origin: str,
    destination: str,
    guest_count: str,
    vibe: str,
    music_genre: str,
    runtime: ToolRuntime,
) -> Command:
    """Update the central wedding state when details are known.
    This tool must complete before calling search specialists.
    """
    return Command(
        update={
            "origin": origin,
            "destination": destination,
            "guest_count": guest_count,
            "vibe": vibe,
            "music_genre": music_genre,
            "messages": [
                ToolMessage(
                    "Successfully locked wedding details into central state.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


print("🛠️ State-reading delegation tools defined!")

🛠️ State-reading delegation tools defined!


In [15]:
COORDINATOR_SYSTEM_PROMPT = """You are the Lead Wedding Coordinator.
1. When given a request, first call `update_wedding_state` to lock in the origin, destination, guest count, vibe, and music genre.
2. Once the state is updated, delegate tasks to your specialists:
   - Call `search_flights`
   - Call `search_venues`
   - Call `suggest_playlist`
3. Synthesize their findings into a luxurious, beautifully structured Master Wedding Proposal.
"""

wedding_memory = InMemorySaver()

coordinator = create_agent(
    model=model,
    tools=[
        update_wedding_state,
        search_flights,
        search_venues,
        suggest_playlist,
    ],
    state_schema=WeddingState,
    system_prompt=COORDINATOR_SYSTEM_PROMPT,
    checkpointer=wedding_memory,
)

print("👑 Enterprise Coordinator Agent ready!")

👑 Enterprise Coordinator Agent ready!


In [ ]:
wedding_request = HumanMessage(
    content="""
We are planning a destination wedding!
- Flying from: London (LHR)
- Destination: Amalfi Coast, Italy (fly into Naples NAP)
- Guest count: 40 guests
- Vibe: Cliffside luxury with sunset outdoor dining
- Music genre: Acoustic pop for ceremony, 80s dance & Italian retro lounge for reception

Please assemble our complete wedding proposal!
"""
)

config = {
    "configurable": {"thread_id": "amalfi_wedding_1"},
    "recursion_limit": 40,
    "tags": ["WeddingPlanner", "Module2Capstone"],
}

print("💍 Planning the Amalfi Wedding (calling subagents)...")
response = await coordinator.ainvoke(
    {"messages": [wedding_request]}, config=config
)

print("\n" + "=" * 60)
print("💐 MASTER WEDDING PROPOSAL")
print("=" * 60)
ans = response["messages"][-1].content
print(ans[0]["text"] if isinstance(ans, list) else ans)

In [ ]:
follow_up = HumanMessage(
    content="""
This is gorgeous! Could we make one change:
What if we add a welcome boat cruise along the coast for our 40 guests the day before the wedding?
Can you recommend how to incorporate that into our itinerary and estimated budget?
"""
)

print("🔄 Refining the proposal with memory...")
response_2 = await coordinator.ainvoke(
    {"messages": [follow_up]}, config=config
)

print("\n" + "=" * 60)
print("💐 UPDATED WEDDING PROPOSAL")
print("=" * 60)
ans_2 = response_2["messages"][-1].content
print(ans_2[0]["text"] if isinstance(ans_2, list) else ans_2)